# 任务一：植物分类 —— 多架构深度集成

**方案（在原 ResNet50 方案 0.9021 基础上全面升级）：**

1. **多架构集成**：ResNet50 + EfficientNetV2-S + DenseNet121（架构差异大，集成多样性远优于同模型多种子）
2. **高分辨率**：训练/验证 320×320，捕捉幼苗形态细节
3. **分层划分**：Stratified 85/15 验证集，保证各类比例一致
4. **两阶段训练**：分类头热身 → 全网络差异化学习率微调（骨干 1e-4 / 头部 3e-4）
5. **训练技巧**：AMP 混合精度、warmup + cosine 退火、label smoothing、MixUp、类别平衡采样
6. **强数据增强**：RandomResizedCrop、双向翻转、旋转、ColorJitter、AutoContrast、Sharpness、RandomErasing
7. **TTA**：3 个尺寸（300/320/352）× 水平翻转，共 6 形态平均
8. **评估指标**：macro-F1（与排行榜一致）


In [ ]:
import os, glob, random, time, copy
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

SEED = 42
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch:", torch.__version__)
if DEVICE == "cuda":
    print("gpu:", torch.cuda.get_device_name(0),
          f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ---- 自动探测数据路径（Kaggle 挂载路径）----
DATA_ROOT = None
for p in glob.glob("/kaggle/input/**/train", recursive=True):
    DATA_ROOT = os.path.dirname(p)
    break
assert DATA_ROOT is not None, "未找到 train 目录，请确认已挂载比赛数据集"
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
TEST_DIR  = os.path.join(DATA_ROOT, "test")
TMPL_CSV  = glob.glob("/kaggle/input/**/*.csv", recursive=True)[0]

# ---- 超参数 ----
IMG_SIZE   = 320
BATCH      = 32
NUM_WORKERS = 2
HEAD_EPOCHS = 2
FT_EPOCHS  = 14
WARMUP_EP  = 1
MIXUP_P    = 0.4
MIXUP_ALPHA = 0.2

# ---- 数据探查 ----
full_probe = datasets.ImageFolder(TRAIN_DIR)
CLASSES = full_probe.classes
targets_all = np.array(full_probe.targets)
print("\n类别（%d）:" % len(CLASSES), CLASSES)
for i, c in enumerate(CLASSES):
    print(f"  {c}: {(targets_all==i).sum()} 张")
test_files = sorted(glob.glob(TEST_DIR + "/*.png") + glob.glob(TEST_DIR + "/*.jpg"))
tmpl_df = pd.read_csv(TMPL_CSV)
print("\n训练图像:", len(full_probe),
      "| 测试图像:", len(test_files),
      "| 模板行数:", len(tmpl_df),
      "| 模板列:", list(tmpl_df.columns))


In [ ]:
from sklearn.model_selection import train_test_split

# 分层划分
train_idx, val_idx = train_test_split(
    np.arange(len(targets_all)), test_size=0.15,
    random_state=SEED, stratify=targets_all)
print("train/val:", len(train_idx), len(val_idx))

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(25),
    transforms.ColorJitter(0.25, 0.25, 0.25, 0.05),
    transforms.RandomAutocontrast(p=0.3),
    transforms.RandomAdjustSharpness(1.5, p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_base = datasets.ImageFolder(TRAIN_DIR, transform=train_tf)
val_base   = datasets.ImageFolder(TRAIN_DIR, transform=val_tf)
train_ds = Subset(train_base, train_idx)
val_ds   = Subset(val_base, val_idx)

# 类别不平衡时自动启用平衡采样
tr_targets = targets_all[train_idx]
counts = np.bincount(tr_targets, minlength=len(CLASSES))
if counts.max() / counts.min() > 1.25:
    sample_w = (1.0 / counts)[tr_targets]
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)
    train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                              num_workers=NUM_WORKERS, pin_memory=True)
    print("启用 WeightedRandomSampler | 类别计数:", counts.tolist())
else:
    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
    print("类别均衡，普通 shuffle | 类别计数:", counts.tolist())
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)


In [ ]:
from torchvision.models import (
    ResNet50_Weights, EfficientNet_V2_S_Weights, DenseNet121_Weights)

def build_model(name, ncls):
    if name == "resnet50":
        m = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, ncls)
    elif name == "effv2s":
        m = models.efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, ncls)
    elif name == "densenet121":
        m = models.densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, ncls)
    else:
        raise ValueError(name)
    return m

def head_parameters(model, name):
    if name == "resnet50":
        return list(model.fc.parameters())
    return list(model.classifier.parameters())


In [ ]:
from sklearn.metrics import f1_score

def evaluate(model, loader):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for x, y in loader:
            preds += model(x.to(DEVICE)).argmax(1).cpu().tolist()
            gts += y.tolist()
    return f1_score(gts, preds, average="macro")

AMP_ENABLED = (DEVICE == "cuda")

def train_model(name, seed):
    seed_everything(seed)
    model = build_model(name, len(CLASSES)).to(DEVICE)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

    # ---- Stage 1：分类头热身 ----
    for p in model.parameters():
        p.requires_grad = False
    head = head_parameters(model, name)
    for p in head:
        p.requires_grad = True
    opt = optim.AdamW(head, lr=1e-3, weight_decay=1e-4)
    print(f"[{name}] stage1 头部热身 ×{HEAD_EPOCHS}")
    for ep in range(HEAD_EPOCHS):
        model.train(); t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        f1 = evaluate(model, val_loader)
        print(f"  head ep{ep+1} f1={f1:.4f} {time.time()-t0:.0f}s", flush=True)

    # ---- Stage 2：全网络微调（差异化学习率 + warmup/cosine）----
    for p in model.parameters():
        p.requires_grad = True
    head_ids = set(id(p) for p in head_parameters(model, name))
    head_p = [p for p in model.parameters() if id(p) in head_ids]
    back_p = [p for p in model.parameters() if id(p) not in head_ids]
    opt = optim.AdamW([
        {"params": back_p, "lr": 1e-4},
        {"params": head_p, "lr": 3e-4},
    ], weight_decay=1e-4)

    def lr_lambda(ep):
        if ep < WARMUP_EP:
            return (ep + 1) / WARMUP_EP
        prog = (ep - WARMUP_EP) / max(1, FT_EPOCHS - WARMUP_EP)
        return 0.5 * (1 + np.cos(np.pi * prog))
    sched = optim.lr_scheduler.LambdaLR(opt, lr_lambda)

    best_f1, best_state = 0.0, None
    print(f"[{name}] stage2 全网络微调 ×{FT_EPOCHS}")
    for ep in range(FT_EPOCHS):
        model.train(); t0 = time.time(); run_loss, nb = 0.0, 0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            do_mix = np.random.rand() < MIXUP_P
            if do_mix:
                lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
                perm = torch.randperm(x.size(0), device=DEVICE)
                x_in = lam * x + (1 - lam) * x[perm]
            else:
                x_in = x
            opt.zero_grad()
            with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                logits = model(x_in)
                if do_mix:
                    loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y[perm])
                else:
                    loss = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            run_loss += loss.item(); nb += 1
        sched.step()
        f1 = evaluate(model, val_loader)
        print(f"  ft ep{ep+1}/{FT_EPOCHS} f1={f1:.4f} loss={run_loss/nb:.3f} "
              f"{time.time()-t0:.0f}s", flush=True)
        if f1 > best_f1:
            best_f1 = f1
            best_state = copy.deepcopy(model.state_dict())

    ckpt = f"/kaggle/working/ckpt_{name}.pth"
    torch.save({"state_dict": best_state, "classes": CLASSES,
                "f1": best_f1, "arch": name}, ckpt)
    print(f"[{name}] best val f1={best_f1:.4f} -> {ckpt}")
    return ckpt, best_f1


In [ ]:
MODEL_NAMES = ["resnet50", "effv2s", "densenet121"]
SEEDS = [42, 0, 7]

results = []
for name, sd in zip(MODEL_NAMES, SEEDS):
    print(f"\n========== {name} (seed={sd}) ==========")
    ckpt, f1 = train_model(name, sd)
    results.append((name, ckpt, f1))

print("\n===== 各模型验证集 macro-F1 =====")
for n, _, f in results:
    print(f"  {n}: {f:.4f}")


In [ ]:
# 在验证集上检验“三模型集成”的增益
val_gt = targets_all[val_idx]
ens_probs = torch.zeros(len(val_ds), len(CLASSES))
for name, ckpt, _ in results:
    d = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    m = build_model(name, len(CLASSES)).to(DEVICE)
    m.load_state_dict(d["state_dict"]); m.eval()
    with torch.no_grad():
        p = torch.cat([torch.softmax(m(x.to(DEVICE)), 1).cpu()
                       for x, _ in val_loader])
    ens_probs += p / len(results)
    print(f"  {name} val probs done")
ens_f1 = f1_score(val_gt, ens_probs.argmax(1).tolist(), average="macro")
print(f"\n>>> 验证集三模型集成 macro-F1 = {ens_f1:.4f}")


In [ ]:
class TestDataset(Dataset):
    def __init__(self, files, tf):
        self.files = files; self.tf = tf
    def __len__(self):
        return len(self.files)
    def __getitem__(self, i):
        return self.tf(Image.open(self.files[i]).convert("RGB"))

print("测试文件数:", len(test_files))
TTA_SIZES = [300, 320, 352]
probs_sum = torch.zeros(len(test_files), len(CLASSES))

for name, ckpt, _ in results:
    d = torch.load(ckpt, map_location=DEVICE, weights_only=False)
    model = build_model(name, len(CLASSES)).to(DEVICE)
    model.load_state_dict(d["state_dict"]); model.eval()
    for sz in TTA_SIZES:
        for flip in (False, True):
            tfs = [transforms.Resize((sz, sz))]
            if flip:
                tfs.append(transforms.RandomHorizontalFlip(p=1.0))
            tfs += [transforms.ToTensor(), transforms.Normalize(MEAN, STD)]
            dl = DataLoader(TestDataset(test_files, transforms.Compose(tfs)),
                            batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
            parts = []
            with torch.no_grad():
                for x in dl:
                    with torch.amp.autocast("cuda", enabled=AMP_ENABLED):
                        parts.append(torch.softmax(model(x.to(DEVICE)), 1).cpu())
            probs_sum += torch.cat(parts) / (len(TTA_SIZES) * 2)
    print(f"  {name} TTA 完成", flush=True)

avg_probs = probs_sum / len(results)
preds = avg_probs.argmax(1).tolist()
print("\n集成预测类别分布:")
for c, n in Counter(CLASSES[p] for p in preds).most_common():
    print(f"  {c}: {n}")


In [ ]:
# 中文文件夹名 -> 模板中的标准英文类别名（与 DATA 中 submission 模板保持一致）
ZH2EN = {
    "黑草": "Black-grass",
    "普通小麦": "Common wheat",
    "宽松丝滑的弯曲": "Loose Silky-bent",
    "无味五月草": "Scentless Mayweed",
    "甜菜": "Sugar beet",
}
VALID_LABELS = set(ZH2EN.values())

file_to_label = {}
for f, p in zip(test_files, preds):
    zh = CLASSES[p]
    file_to_label[os.path.basename(f)] = ZH2EN.get(zh, zh)

out = tmpl_df.copy()
def lookup(x):
    if x in file_to_label:
        return file_to_label[x]
    return file_to_label.get(os.path.splitext(x)[0] + ".png")
out["Category"] = out["ID"].map(lookup)

# ---- 完整性校验 ----
assert out["Category"].notna().all(), "存在未映射的 ID！"
assert set(out["Category"].unique()) <= VALID_LABELS, "存在非法标签！"
assert len(out) == len(tmpl_df), "行数与模板不一致！"

out_path = "/kaggle/working/submission.csv"
out.to_csv(out_path, index=False)
print("已保存:", out_path, "| 行数:", len(out))
print(out.head(10))
print("\n最终提交分布:")
print(out["Category"].value_counts())
print("\n=== DONE ===")
